In [ ]:
# Core imports
import os
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from PIL import Image
import cv2

# Set paths
PROJECT_ROOT = Path(os.getcwd()).parent if 'notebooks' in os.getcwd() else Path(os.getcwd())
DATA_DIR = PROJECT_ROOT / 'data'
SAMPLES_DIR = DATA_DIR / 'samples'
BCCD_DIR = DATA_DIR / 'bccd'
KAGGLE_DIR = DATA_DIR / 'kaggle'

print(f"Project root: {PROJECT_ROOT}")
print(f"Data directory: {DATA_DIR}")

## 1. Explore Your Sample Image

In [ ]:
# Load your sample image
sample_path = SAMPLES_DIR / '1000026798.jpg'

if sample_path.exists():
    img = Image.open(sample_path)
    img_array = np.array(img)
    
    print(f"Image shape: {img_array.shape}")
    print(f"Image size: {img.size} (width x height)")
    print(f"Image mode: {img.mode}")
    print(f"Data type: {img_array.dtype}")
    print(f"Value range: [{img_array.min()}, {img_array.max()}]")
    
    # Display the image
    plt.figure(figsize=(12, 16))
    plt.imshow(img_array)
    plt.title('Your Blood Smear Sample')
    plt.axis('off')
    plt.show()
else:
    print(f"Sample image not found at {sample_path}")

In [ ]:
# Analyze color distribution (helpful for understanding staining)
if sample_path.exists():
    fig, axes = plt.subplots(1, 3, figsize=(15, 4))
    colors = ['red', 'green', 'blue']
    
    for i, (ax, color) in enumerate(zip(axes, colors)):
        ax.hist(img_array[:,:,i].ravel(), bins=256, color=color, alpha=0.7)
        ax.set_title(f'{color.upper()} Channel')
        ax.set_xlim([0, 255])
    
    plt.suptitle('Color Channel Distributions')
    plt.tight_layout()
    plt.show()

In [ ]:
# Zoom into a section to see individual cells
if sample_path.exists():
    # Take a 400x400 crop from the center
    h, w = img_array.shape[:2]
    cx, cy = w // 2, h // 2
    crop_size = 400
    
    crop = img_array[cy-crop_size//2:cy+crop_size//2, cx-crop_size//2:cx+crop_size//2]
    
    fig, axes = plt.subplots(1, 2, figsize=(14, 7))
    
    # Full image with crop box
    axes[0].imshow(img_array)
    rect = plt.Rectangle((cx-crop_size//2, cy-crop_size//2), crop_size, crop_size, 
                          fill=False, color='yellow', linewidth=2)
    axes[0].add_patch(rect)
    axes[0].set_title('Full Image (crop area in yellow)')
    axes[0].axis('off')
    
    # Cropped view
    axes[1].imshow(crop)
    axes[1].set_title('Zoomed View - Individual Cells')
    axes[1].axis('off')
    
    plt.tight_layout()
    plt.show()

## 2. Download BCCD Dataset (for Cell Detection)

In [ ]:
# Check if BCCD is already downloaded
bccd_images = BCCD_DIR / 'BCCD' / 'JPEGImages'
bccd_annotations = BCCD_DIR / 'BCCD' / 'Annotations'

if bccd_images.exists():
    images = list(bccd_images.glob('*.jpg'))
    annotations = list(bccd_annotations.glob('*.xml'))
    print(f"BCCD Dataset found!")
    print(f"  Images: {len(images)}")
    print(f"  Annotations: {len(annotations)}")
else:
    print("BCCD Dataset not found. Run this command in terminal:")
    print(f"")
    print(f"  cd {BCCD_DIR}")
    print(f"  git clone https://github.com/Shenggan/BCCD_Dataset.git BCCD")
    print(f"")

In [ ]:
# If BCCD is downloaded, explore it
if bccd_images.exists():
    import xml.etree.ElementTree as ET
    
    # Parse an annotation file to understand the format
    sample_xml = list(bccd_annotations.glob('*.xml'))[0]
    tree = ET.parse(sample_xml)
    root = tree.getroot()
    
    print(f"Sample annotation: {sample_xml.name}")
    print(f"Image filename: {root.find('filename').text}")
    print(f"Image size: {root.find('size/width').text}x{root.find('size/height').text}")
    print(f"\nObjects in image:")
    
    for obj in root.findall('object'):
        name = obj.find('name').text
        bbox = obj.find('bndbox')
        xmin = bbox.find('xmin').text
        ymin = bbox.find('ymin').text
        xmax = bbox.find('xmax').text
        ymax = bbox.find('ymax').text
        print(f"  - {name}: ({xmin}, {ymin}) to ({xmax}, {ymax})")

In [ ]:
# Visualize a BCCD sample with annotations
if bccd_images.exists():
    # Load image and its annotation
    sample_img_path = list(bccd_images.glob('*.jpg'))[0]
    sample_xml_path = bccd_annotations / (sample_img_path.stem + '.xml')
    
    img = cv2.imread(str(sample_img_path))
    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    
    # Parse annotations
    tree = ET.parse(sample_xml_path)
    root = tree.getroot()
    
    # Draw bounding boxes
    colors = {'RBC': (255, 0, 0), 'WBC': (0, 0, 255), 'Platelets': (0, 255, 0)}
    
    for obj in root.findall('object'):
        name = obj.find('name').text
        bbox = obj.find('bndbox')
        xmin = int(bbox.find('xmin').text)
        ymin = int(bbox.find('ymin').text)
        xmax = int(bbox.find('xmax').text)
        ymax = int(bbox.find('ymax').text)
        
        color = colors.get(name, (255, 255, 0))
        cv2.rectangle(img, (xmin, ymin), (xmax, ymax), color, 2)
        cv2.putText(img, name, (xmin, ymin-5), cv2.FONT_HERSHEY_SIMPLEX, 0.5, color, 1)
    
    plt.figure(figsize=(12, 10))
    plt.imshow(img)
    plt.title('BCCD Sample with Annotations (RBC=Red, WBC=Blue, Platelets=Green)')
    plt.axis('off')
    plt.show()

In [ ]:
# Count class distribution in BCCD
if bccd_annotations.exists():
    class_counts = {'RBC': 0, 'WBC': 0, 'Platelets': 0}
    
    for xml_file in bccd_annotations.glob('*.xml'):
        tree = ET.parse(xml_file)
        root = tree.getroot()
        for obj in root.findall('object'):
            name = obj.find('name').text
            if name in class_counts:
                class_counts[name] += 1
    
    print("BCCD Class Distribution:")
    for cls, count in class_counts.items():
        print(f"  {cls}: {count}")
    
    # Plot
    plt.figure(figsize=(8, 5))
    plt.bar(class_counts.keys(), class_counts.values(), color=['red', 'blue', 'green'])
    plt.title('BCCD Dataset - Cell Type Distribution')
    plt.ylabel('Count')
    plt.show()

## 3. Download Kaggle Dataset (for WBC Classification)

In [ ]:
# Check if Kaggle dataset is downloaded
kaggle_train = KAGGLE_DIR / 'dataset2-master' / 'dataset2-master' / 'images' / 'TRAIN'
kaggle_test = KAGGLE_DIR / 'dataset2-master' / 'dataset2-master' / 'images' / 'TEST'

# Alternative path structure
if not kaggle_train.exists():
    kaggle_train = KAGGLE_DIR / 'dataset-master' / 'dataset-master' / 'JPEGImages'
    
if not kaggle_train.exists():
    kaggle_train = KAGGLE_DIR / 'images' / 'TRAIN'

if kaggle_train.exists():
    classes = [d.name for d in kaggle_train.iterdir() if d.is_dir()]
    print(f"Kaggle Dataset found!")
    print(f"Classes: {classes}")
    
    for cls in classes:
        count = len(list((kaggle_train / cls).glob('*.jpeg'))) + len(list((kaggle_train / cls).glob('*.jpg')))
        print(f"  {cls}: {count} images")
else:
    print("Kaggle Dataset not found. To download:")
    print("")
    print("1. Install kaggle CLI: pip install kaggle")
    print("2. Set up API key: https://www.kaggle.com/docs/api")
    print("3. Run:")
    print(f"   kaggle datasets download -d paultimothymooney/blood-cells -p {KAGGLE_DIR} --unzip")
    print("")
    print("Or download manually from:")
    print("https://www.kaggle.com/datasets/paultimothymooney/blood-cells")

In [ ]:
# If Kaggle is downloaded, visualize samples
if kaggle_train.exists():
    classes = [d.name for d in kaggle_train.iterdir() if d.is_dir()]
    
    fig, axes = plt.subplots(2, 4, figsize=(16, 8))
    axes = axes.ravel()
    
    for i, cls in enumerate(classes):
        cls_dir = kaggle_train / cls
        images = list(cls_dir.glob('*.jpeg')) + list(cls_dir.glob('*.jpg'))
        if images:
            # Show 2 samples per class
            for j in range(2):
                if i*2+j < len(axes) and j < len(images):
                    img = Image.open(images[j])
                    axes[i*2+j].imshow(img)
                    axes[i*2+j].set_title(cls)
                    axes[i*2+j].axis('off')
    
    plt.suptitle('Kaggle Blood Cell Images - WBC Subtypes')
    plt.tight_layout()
    plt.show()

## 4. Compare Your Image with Training Data

In [ ]:
# Side-by-side comparison
fig, axes = plt.subplots(1, 3, figsize=(18, 6))

# Your sample
if sample_path.exists():
    img = Image.open(sample_path)
    axes[0].imshow(img)
    axes[0].set_title(f'Your Sample\n{img.size[0]}x{img.size[1]}')
    axes[0].axis('off')

# BCCD sample
if bccd_images.exists():
    bccd_sample = list(bccd_images.glob('*.jpg'))[0]
    img = Image.open(bccd_sample)
    axes[1].imshow(img)
    axes[1].set_title(f'BCCD Sample\n{img.size[0]}x{img.size[1]}')
    axes[1].axis('off')
else:
    axes[1].text(0.5, 0.5, 'BCCD not downloaded', ha='center', va='center')
    axes[1].set_title('BCCD Sample')

# Kaggle sample
if kaggle_train.exists():
    classes = [d for d in kaggle_train.iterdir() if d.is_dir()]
    if classes:
        kaggle_sample = list((classes[0]).glob('*.jpeg'))[0] if list((classes[0]).glob('*.jpeg')) else list((classes[0]).glob('*.jpg'))[0]
        img = Image.open(kaggle_sample)
        axes[2].imshow(img)
        axes[2].set_title(f'Kaggle Sample ({classes[0].name})\n{img.size[0]}x{img.size[1]}')
        axes[2].axis('off')
else:
    axes[2].text(0.5, 0.5, 'Kaggle not downloaded', ha='center', va='center')
    axes[2].set_title('Kaggle Sample')

plt.suptitle('Image Comparison: Your Sample vs Training Datasets')
plt.tight_layout()
plt.show()

## 5. GPU Check

In [ ]:
import torch

print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")

if torch.cuda.is_available():
    print(f"CUDA version: {torch.version.cuda}")
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"GPU Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
else:
    print("\n⚠️ CUDA not available! Training will be slow on CPU.")

## Next Steps

1. **Download the datasets** (if not already done)
2. Run `02_train_detector.ipynb` to train YOLOv8 on BCCD
3. Run `03_train_classifier.ipynb` to train WBC classifier on Kaggle
4. Run `04_full_pipeline.ipynb` to test on your images!